# Functional array updates with .at[]

JAX arrays are immutable: `x[i] = v` raises an error. Updates go through the `.at[...]` property instead, which returns a **new** array and leaves the old one untouched. Under `jit`, XLA usually turns these copies back into in-place writes, so the functional style costs nothing.

Write `scatter_three_ways(grid, idx, vals)`:

- `grid`: `(H, W)` float array.
- `idx`: `(N, 2)` int array of `(row, col)` pairs, all in range. **The same cell can appear more than once.**
- `vals`: `(N,)` float array; `vals[n]` is aimed at cell `idx[n]`.

Return a tuple of three float32 `(H, W)` arrays:

1. `added`: `grid` with every `vals[n]` added to its cell. A cell listed twice receives both updates.
2. `peaked`: `grid` where each listed cell becomes the maximum of its old value and every value aimed at it. Unlisted cells are unchanged.
3. `framed`: `grid` with its first row and last column set to `0.0`, everything else unchanged (this one ignores `idx`).

Use `jax.numpy` and `.at[]`; no Python loops over `N`.

This problem runs in **Colab** (JAX isn't available in the browser). Use the default CPU runtime. JAX computes in float32 here (64-bit mode is off), so NumPy float64 inputs become float32 inside `jnp`. The notebook includes the same tests.

Write your solution in the next cell, then run the test cell.

In [ ]:
import jax.numpy as jnp

def scatter_three_ways(grid, idx, vals):
    """grid (H, W), idx (N, 2) int, vals (N,). Return (added, peaked, framed), each (H, W) float32."""
    raise NotImplementedError


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [[0.0, 0.0, 0.0], [0.0, 0.0, 0.0]], "dtype": "float64"}, {"__nd__": [[0, 1], [1, 2], [0, 1]], "dtype": "int64"}, {"__nd__": [1.0, 2.0, 5.0], "dtype": "float64"}], "e": {"__tup__": [{"__nd__": [[0.0, 6.0, 0.0], [0.0, 0.0, 2.0]], "dtype": "float32"}, {"__nd__": [[0.0, 5.0, 0.0], [0.0, 0.0, 2.0]], "dtype": "float32"}, {"__nd__": [[0.0, 0.0, 0.0], [0.0, 0.0, 0.0]], "dtype": "float32"}]}}, {"a": [{"__nd__": [[0.0, 1.0, 2.0], [3.0, 4.0, 5.0]], "dtype": "float64"}, {"__nd__": [[1, 0], [1, 0]], "dtype": "int64"}, {"__nd__": [-4.0, 10.0], "dtype": "float64"}], "e": {"__tup__": [{"__nd__": [[0.0, 1.0, 2.0], [9.0, 4.0, 5.0]], "dtype": "float32"}, {"__nd__": [[0.0, 1.0, 2.0], [10.0, 4.0, 5.0]], "dtype": "float32"}, {"__nd__": [[0.0, 0.0, 0.0], [3.0, 4.0, 0.0]], "dtype": "float32"}]}}, {"a": [{"__nd__": [[1.0]], "dtype": "float64"}, {"__nd__": [[0, 0], [0, 0], [0, 0]], "dtype": "int64"}, {"__nd__": [0.5, -2.0, 0.25], "dtype": "float64"}], "e": {"__tup__": [{"__nd__": [[-0.25]], "dtype": "float32"}, {"__nd__": [[1.0]], "dtype": "float32"}, {"__nd__": [[0.0]], "dtype": "float32"}]}}, {"a": [{"__nd__": [[-1.0, -1.0], [-1.0, -1.0], [-1.0, -1.0]], "dtype": "float64"}, {"__nd__": [[2, 1], [0, 0]], "dtype": "int64"}, {"__nd__": [-5.0, -0.5], "dtype": "float64"}], "e": {"__tup__": [{"__nd__": [[-1.5, -1.0], [-1.0, -1.0], [-1.0, -6.0]], "dtype": "float32"}, {"__nd__": [[-0.5, -1.0], [-1.0, -1.0], [-1.0, -1.0]], "dtype": "float32"}, {"__nd__": [[0.0, 0.0], [-1.0, 0.0], [-1.0, 0.0]], "dtype": "float32"}]}}, {"a": [{"__nd__": [[2.0409191213851825, -2.5556650313141818, 0.41809884672577885, -0.5677696061279298, -0.45264929211044586], [-0.2155971630897659, -2.019986129147251, -0.23193237764418947, -0.8652130762749417, 3.3229995166448827], [0.22578661322792176, -0.3526307943415954, -0.2812874181513504, -0.6680463461089501, -1.0551505512051214], [-0.39080097723465473, 0.48194538850678587, -0.2385536065733667, 0.9577587029597641, -0.19980212906658]], "dtype": "float64"}, {"__nd__": [[3, 3], [0, 0], [0, 0], [3, 3], [3, 3], [1, 1], [0, 0], [1, 1], [0, 0], [3, 3], [2, 2], [2, 2]], "dtype": "int64"}, {"__nd__": [1.9350880340988528, -0.2696203273419135, -0.24355867907910456, 1.0023136012756912, -0.8864599431605871, -0.291720232439864, 0.8825389674564839, 0.5803500161908991, 0.09151670328235219, 0.6701043548284794, -2.8281623068437627, 1.02130681750008], "dtype": "float64"}], "e": {"__tup__": [{"__nd__": [[2.501795768737793, -2.5556650161743164, 0.4180988371372223, -0.5677695870399475, -0.4526492953300476], [-0.21559716761112213, -1.7313562631607056, -0.2319323718547821, -0.8652130961418152, 3.3229994773864746], [0.22578661143779755, -0.3526307940483093, -2.0881428718566895, -0.6680463552474976, -1.0551505088806152], [-0.39080098271369934, 0.4819453954696655, -0.23855361342430115, 3.678804874420166, -0.19980213046073914]], "dtype": "float32"}, {"__nd__": [[2.040919065475464, -2.5556650161743164, 0.4180988371372223, -0.5677695870399475, -0.4526492953300476], [-0.21559716761112213, 0.5803500413894653, -0.2319323718547821, -0.8652130961418152, 3.3229994773864746], [0.22578661143779755, -0.3526307940483093, 1.0213068723678589, -0.6680463552474976, -1.0551505088806152], [-0.39080098271369934, 0.4819453954696655, -0.23855361342430115, 1.935088038444519, -0.19980213046073914]], "dtype": "float32"}, {"__nd__": [[0.0, 0.0, 0.0, 0.0, 0.0], [-0.21559716761112213, -2.019986152648926, -0.2319323718547821, -0.8652130961418152, 0.0], [0.22578661143779755, -0.3526307940483093, -0.28128743171691895, -0.6680463552474976, 0.0], [-0.39080098271369934, 0.4819453954696655, -0.23855361342430115, 0.9577587246894836, 0.0]], "dtype": "float32"}]}}, {"a": [{"__nd__": [[-0.9596447598081417, -1.6686198426559695, 0.27644575952099965], [0.7005448853493901, -0.4447674556827841, -1.0764058401008076], [0.026124833534033623, -0.05274730824287927, 1.4055981660180925], [0.7474079874793504, 0.19381564626462, 1.1116332052239921], [-0.20552304990579248, -0.9258995736483681, 0.584058311025248], [0.5825384186556901, -0.2148289111268558, -0.7828085779639662]], "dtype": "float64"}, {"__nd__": [[2, 1], [2, 0], [0, 2], [3, 2], [5, 2], [1, 1], [4, 2], [1, 1], [0, 1], [4, 2], [1, 1], [4, 2], [1, 1], [3, 2], [5, 2], [5, 0], [2, 1], [1, 1], [1, 0], [5, 0]], "dtype": "int64"}, {"__nd__": [0.354531946286926, -1.0484055185445111, 1.4059629431348852, -0.021651229055558368, -0.3722505640006159, -1.7181849497326165, 1.6818255450666806, 0.7527785926973876, 0.753563837509362, 1.1378812589177814, 0.3492265781230293, -0.6392466105764212, -0.8002412270301018, -0.800199979361005, 1.3700723413337117, -1.4603812011954127, -0.5963695117707888, -0.32124391928619556, 0.22461902534909414, 0.5753493885078089], "dtype": "float64"}], "e": {"__tup__": [{"__nd__": [[-0.9596447348594666, -0.915056049823761, 1.6824086904525757], [0.9251639246940613, -2.1824324131011963, -1.0764058828353882], [-1.0222806930541992, -0.29458487033843994, 1.4055981636047363], [0.7474079728126526, 0.19381564855575562, 0.28978192806243896], [-0.2055230438709259, -0.9258995652198792, 2.7645187377929688], [-0.3024933338165283, -0.21482890844345093, 0.2150132656097412]], "dtype": "float32"}, {"__nd__": [[-0.9596447348594666, 0.7535638213157654, 1.4059629440307617], [0.7005448937416077, 0.7527785897254944, -1.0764058828353882], [0.026124833151698112, 0.3545319437980652, 1.4055981636047363], [0.7474079728126526, 0.19381564855575562, 1.1116331815719604], [-0.2055230438709259, -0.9258995652198792, 1.6818255186080933], [0.5825384259223938, -0.21482890844345093, 1.370072364807129]], "dtype": "float32"}, {"__nd__": [[0.0, 0.0, 0.0], [0.7005448937416077, -0.44476744532585144, 0.0], [0.026124833151698112, -0.05274730920791626, 0.0], [0.7474079728126526, 0.19381564855575562, 0.0], [-0.2055230438709259, -0.9258995652198792, 0.0], [0.5825384259223938, -0.21482890844345093, 0.0]], "dtype": "float32"}]}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(scatter_three_ways, json.loads(_TESTS), 'jax', *[1e-05, 1e-06])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")